# For1Sight V2.7 – Machine Learning Development
## Stage 1: Data Exploration and Validation

**Prediction objective:** Predict the Grand Prix winner after qualifying and before the race.

**Chronological evaluation plan:** 2024 training, 2025 validation/model selection, and 2026 final unseen test.

**Agreed exclusions:** the two unmatched 2026 events: Bahrain Grand Prix in Malaysia and United States Grand Prix.

This is a read-only exploration notebook. It does not write to HDFS, rebuild the warehouse, train models, or modify Streamlit.
Run cells from top to bottom.

## 0. Environment setup

Install the Microsoft **Python** and **Jupyter** extensions in VS Code.

In the VS Code terminal, using the Python environment already used by For1Sight, run:

```powershell
python -m pip install ipykernel jupyter pandas pyarrow numpy matplotlib scikit-learn joblib
python -m ipykernel install --user --name for1sight --display-name "Python (For1Sight)"
```

Then select **Python (For1Sight)** as the notebook kernel.

## 1. Check the active Python environment

In [1]:
import sys
import platform
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Operating system:", platform.platform())
print("Current working directory:", Path.cwd())

Python executable: C:\Users\ayepd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe
Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Operating system: Windows-11-10.0.26300-SP0
Current working directory: c:\Users\ayepd\Desktop\For1Sight Project\notebooks


## 2. Set the For1Sight project and dataset paths

The default path matches the project location used previously. Update `PROJECT_ROOT` if your project folder has moved.

In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\ayepd\Desktop\For1Sight Project")
DATA_PATH = PROJECT_ROOT / "for1sight_warehouse" / "prediction_training_v2_6.parquet"

print("Project root exists:", PROJECT_ROOT.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError("Update PROJECT_ROOT to your actual For1Sight folder.")
if not DATA_PATH.exists():
    raise FileNotFoundError("V2.6 local Parquet mirror not found at the path above.")

Project root exists: True
Dataset exists: True
Dataset path: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\prediction_training_v2_6.parquet


## 3. Import libraries and load the V2.6 dataset

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

df = pd.read_parquet(DATA_PATH, engine="pyarrow")
print(f"Dataset loaded: {len(df):,} rows and {len(df.columns):,} columns")
display(df.head(10))

Pandas: 2.3.3
NumPy: 2.5.3
Dataset loaded: 1,354 rows and 49 columns


,dataset_version,year,round,event_date,event_name,silver_event_name,event_norm,driver_code,driver_code_norm,constructor,...,grid_feature_count,winner_label_count,qualifying_coverage,grid_coverage,event_prediction_ready,model_training_eligible,finishing_position,is_winner,race_points,race_status
0,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,VER,VER,Red Bull,...,20,1,1.0,1.0,True,True,1,1,26,Finished
1,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,LEC,LEC,Ferrari,...,20,1,1.0,1.0,True,True,4,0,12,Finished
2,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,RUS,RUS,Mercedes,...,20,1,1.0,1.0,True,True,5,0,10,Finished
3,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,SAI,SAI,Ferrari,...,20,1,1.0,1.0,True,True,3,0,15,Finished
4,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,PER,PER,Red Bull,...,20,1,1.0,1.0,True,True,2,0,18,Finished
5,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,ALO,ALO,Aston Martin,...,20,1,1.0,1.0,True,True,9,0,2,Finished
6,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,NOR,NOR,McLaren,...,20,1,1.0,1.0,True,True,6,0,8,Finished
7,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,PIA,PIA,McLaren,...,20,1,1.0,1.0,True,True,8,0,4,Finished
8,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,HAM,HAM,Mercedes,...,20,1,1.0,1.0,True,True,7,0,6,Finished
9,V2.6,2024,1,2024-03-02,Bahrain Grand Prix,Bahrain Grand Prix,bahrain,HUL,HUL,Haas F1 Team,...,20,1,1.0,1.0,True,True,16,0,0,Lapped


## 4. Inspect the dataset schema

In [4]:
schema_summary = pd.DataFrame({
    "column": df.columns,
    "data_type": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
display(schema_summary)

,column,data_type,missing_count,missing_percent,unique_values
0,dataset_version,object,0,0.00,1
1,year,int32,0,0.00,3
2,round,int32,0,0.00,24
3,event_date,object,0,0.00,66
4,event_name,object,0,0.00,26
5,silver_event_name,object,44,3.25,25
6,event_norm,object,0,0.00,26
7,driver_code,object,0,0.00,28
8,driver_code_norm,object,0,0.00,28
9,constructor,object,0,0.00,12


## 5. Confirm required target and eligibility columns

In [5]:
required_columns = [
    "year", "event_name", "driver_code", "constructor",
    "starting_grid", "finishing_position", "is_winner",
    "event_prediction_ready", "model_training_eligible",
]
missing_required = [c for c in required_columns if c not in df.columns]
print("Required columns found:", len(required_columns) - len(missing_required), "/", len(required_columns))
print("Missing required columns:", missing_required if missing_required else "None")
if missing_required:
    print("STOP: do not train yet. Reconcile the V2.6 schema first.")

Required columns found: 9 / 9
Missing required columns: None


## 6. Inspect season and event coverage

In [6]:
if {"year", "event_name"}.issubset(df.columns):
    agg = {
        "driver_event_rows": ("event_name", "size"),
        "unique_events": ("event_name", "nunique"),
    }
    if "driver_code" in df.columns:
        agg["unique_drivers"] = ("driver_code", "nunique")
    season_summary = df.groupby("year").agg(**agg).reset_index().sort_values("year")
    display(season_summary)

    event_list = (
        df[["year", "event_name"]].drop_duplicates()
        .sort_values(["year", "event_name"]).reset_index(drop=True)
    )
    display(event_list)
else:
    print("Cannot inspect season/event coverage: year or event_name is missing.")

,year,driver_event_rows,unique_events,unique_drivers
0,2024,479,24,24
1,2025,479,24,21
2,2026,396,18,23


,year,event_name
0,2024,Abu Dhabi Grand Prix
1,2024,Australian Grand Prix
2,2024,Austrian Grand Prix
3,2024,Azerbaijan Grand Prix
4,2024,Bahrain Grand Prix
...,...,...
61,2026,Miami Grand Prix
62,2026,Monaco Grand Prix
63,2026,Singapore Grand Prix
64,2026,Spanish Grand Prix


## 7. Verify the two agreed 2026 event exclusions

We exclude only the two unmatched 2026 events. Earlier United States Grand Prix events must remain included if present. This cell displays the actual event names and normalized keys before training.

In [7]:
import re

def normalize_event_name(value):
    value = str(value or "").lower().strip()
    value = re.sub(r"\bgrand\s*prix\b", "", value)
    return re.sub(r"[^a-z0-9]", "", value)

if {"year", "event_name"}.issubset(df.columns):
    event_check = df[["year", "event_name"]].drop_duplicates().copy()
    event_check["event_norm_check"] = event_check["event_name"].map(normalize_event_name)
    keywords = ["bahrain", "malaysia", "united states", "unitedstates"]
    mask = event_check["event_name"].str.lower().str.contains("|".join(keywords), regex=True, na=False)
    display(event_check[mask].sort_values(["year", "event_name"]))
else:
    print("Required columns are missing.")

,year,event_name,event_norm_check
0,2024,Bahrain Grand Prix,bahrain
359,2024,United States Grand Prix,unitedstates
539,2025,Bahrain Grand Prix,bahrain
838,2025,United States Grand Prix,unitedstates
1288,2026,Bahrain Grand Prix in Malaysia,bahraininmalaysia
1332,2026,United States Grand Prix,unitedstates


## 8. Check readiness flags and winner labels

In [8]:
def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})

for col in ["event_prediction_ready", "model_training_eligible"]:
    if col in df.columns:
        print(f"\\n{col}:")
        display(as_bool(df[col]).value_counts(dropna=False).rename("records"))

if {"year", "event_name", "is_winner"}.issubset(df.columns):
    winner_check = df.copy()
    winner_check["is_winner_numeric"] = pd.to_numeric(winner_check["is_winner"], errors="coerce")
    event_winners = (
        winner_check.groupby(["year", "event_name"])["is_winner_numeric"]
        .agg(["sum", "count"]).reset_index()
        .rename(columns={"sum":"winner_label_count", "count":"driver_rows"})
    )
    event_winners["valid_one_winner"] = event_winners["winner_label_count"] == 1
    display(event_winners["valid_one_winner"].value_counts())
    display(event_winners.loc[~event_winners["valid_one_winner"]])

\nevent_prediction_ready:


event_prediction_ready
True     1310
False      44
Name: records, dtype: int64

\nmodel_training_eligible:


model_training_eligible
True     1310
False      44
Name: records, dtype: int64

valid_one_winner
True    66
Name: count, dtype: int64

,year,event_name,winner_label_count,driver_rows,valid_one_winner


## 9. Check candidate model features

In [9]:
candidate_features = [
    "round", "starting_grid",
    "best_practice_lap_s", "best_qualifying_lap_s",
    "best_quali_s1_s", "best_quali_s2_s", "best_quali_s3_s",
    "theoretical_best_sector_sum_s",
    "practice_to_qualifying_improvement_s",
    "practice_delta_to_event_best_s", "qualifying_delta_to_event_best_s",
    "event_best_practice_lap_s", "event_best_qualifying_lap_s",
    "qualifying_rank", "practice_timed_laps", "qualifying_timed_laps",
    "driver_prior_avg_finish_3", "driver_prior_avg_points_3",
    "driver_circuit_prior_avg_finish", "driver_circuit_prior_starts",
    "constructor_prior_avg_finish_3", "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish", "constructor_circuit_prior_starts",
    "has_practice_data", "has_qualifying_data", "has_grid_data",
    "driver_code", "constructor", "event_name",
]
feature_check = pd.DataFrame({
    "feature": candidate_features,
    "available": [c in df.columns for c in candidate_features],
})
display(feature_check)
available_features = feature_check.loc[feature_check["available"], "feature"].tolist()
missing_features = feature_check.loc[~feature_check["available"], "feature"].tolist()
print("Available candidate features:", len(available_features))
print("Missing candidate features:", missing_features)

,feature,available
0,round,True
1,starting_grid,True
2,best_practice_lap_s,True
3,best_qualifying_lap_s,True
4,best_quali_s1_s,True
5,best_quali_s2_s,True
6,best_quali_s3_s,True
7,theoretical_best_sector_sum_s,True
8,practice_to_qualifying_improvement_s,True
9,practice_delta_to_event_best_s,True


Available candidate features: 30
Missing candidate features: []


## 10. Inspect missing values in available candidate features

In [10]:
if available_features:
    feature_missingness = (
        df[available_features].isna().mean().mul(100)
        .sort_values(ascending=False).rename("missing_percent").reset_index()
        .rename(columns={"index":"feature"})
    )
    display(feature_missingness)

,feature,missing_percent
0,driver_circuit_prior_avg_finish,49.556869
1,constructor_circuit_prior_avg_finish,43.353028
2,practice_to_qualifying_improvement_s,3.988183
3,theoretical_best_sector_sum_s,3.914328
4,best_quali_s2_s,3.914328
5,best_quali_s1_s,3.914328
6,qualifying_rank,3.914328
7,qualifying_delta_to_event_best_s,3.914328
8,best_quali_s3_s,3.914328
9,best_qualifying_lap_s,3.914328


## 11. Prepare an eligible-row preview (no model training)

This preview applies the V2.6 readiness flags, requires valid winner/finishing labels, and excludes the two agreed 2026 events. It does not save a new dataset.

In [11]:
if all(c in df.columns for c in required_columns):
    preview = df.copy()
    preview["year"] = pd.to_numeric(preview["year"], errors="coerce")
    preview["is_winner_numeric"] = pd.to_numeric(preview["is_winner"], errors="coerce")
    preview["finishing_position_numeric"] = pd.to_numeric(preview["finishing_position"], errors="coerce")
    preview["event_norm_check"] = preview["event_name"].map(normalize_event_name)
    preview["event_prediction_ready_bool"] = as_bool(preview["event_prediction_ready"])
    preview["model_training_eligible_bool"] = as_bool(preview["model_training_eligible"])

    excluded_2026 = (
        (preview["year"] == 2026)
        & preview["event_norm_check"].isin({"bahraininmalaysia", "unitedstates"})
    )
    eligible_preview = preview.loc[
        ~excluded_2026
        & preview["event_prediction_ready_bool"]
        & preview["model_training_eligible_bool"]
        & preview["finishing_position_numeric"].notna()
        & preview["is_winner_numeric"].isin([0, 1])
    ].copy()

    print("Original rows:", f"{len(df):,}")
    print("Rows excluded by agreed 2026 event rule:", f"{int(excluded_2026.sum()):,}")
    print("Eligible rows in preview:", f"{len(eligible_preview):,}")
    print("Unique eligible events:", eligible_preview[["year", "event_norm_check"]].drop_duplicates().shape[0])

    split_preview = (
        eligible_preview.groupby("year")
        .agg(
            rows=("event_name", "size"),
            events=("event_norm_check", "nunique"),
            winner_labels=("is_winner_numeric", "sum"),
        )
        .reset_index().sort_values("year")
    )
    display(split_preview)
else:
    print("Cannot create eligibility preview because required columns are missing.")

Original rows: 1,354
Rows excluded by agreed 2026 event rule: 44
Eligible rows in preview: 1,310
Unique eligible events: 64


,year,rows,events,winner_labels
0,2024,479,24,24
1,2025,479,24,24
2,2026,352,16,16


## 12. Leakage prevention

The target is `is_winner`. Never include these as model inputs:
- `is_winner`
- `finishing_position`
- final race points
- race status / DNF result
- race-session telemetry
- any feature calculated using the current or future race result

Historical driver, constructor and circuit statistics must use previous events only.

Keep the chronological split: 2024 training, 2025 validation/model selection, and 2026 final unseen test. Do not randomly split driver rows; all drivers from the same Grand Prix must stay together.

## 13. Stage 1 completion checklist

- [ ] Correct `Python (For1Sight)` kernel selected.
- [ ] V2.6 Parquet file loads.
- [ ] Required target and eligibility columns are present.
- [ ] Two excluded 2026 events match the actual event names.
- [ ] Every included Grand Prix has exactly one winner label.
- [ ] Feature availability and missingness are understood.
- [ ] Chronological split has records for 2024, 2025 and 2026.

**Next:** Stage 2 – prepare the model feature matrix and train Logistic Regression, Random Forest and Gradient Boosting.

Do not run the final V2.7 training script until we have reviewed this notebook's outputs.